Connect to localhost:9200 of elasticSearch

In [1]:
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200",
                   basic_auth = ('elasticsearch', 'e3TKzHmKRFWBP4gY--cjeQ'),
                   request_timeout=60,
                   )
es.ping() 
print(es.info())


{'name': 'LAPTOP-ANN0J427', 'cluster_name': 'elasticsearch', 'cluster_uuid': '6AxnonBCTU-8fGsKa_EYMQ', 'version': {'number': '9.1.3', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '0c781091a2f57de895a73a1391ff8426c0153c8d', 'build_date': '2025-08-24T22:05:04.526302670Z', 'build_snapshot': False, 'lucene_version': '10.2.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [2]:
import pandas as pd
df = pd.read_csv('../../preprocessing/cleanedDataset.csv')
print(df.shape)
df.head()


(64, 5)


,scheme_id,site,scheme_name,description,scheme_link
0,1,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,आनंदाचा शिधा,दि. 04.10.2022 च्या शासन निर्णयानुसार राष्ट्री...,https://mahafood.gov.in/scheme/%e0%a4%86%e0%a4...
1,2,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,एपीएल शेतकरी,"राज्यातील छत्रपती संभाजीनगर, जालना, नांदेड, बी...",https://mahafood.gov.in/scheme/%e0%a4%8f%e0%a4...
2,3,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,शिवभोजन,राज्यातील गरीब व गरजू जनतेला सवलतीच्या दरात भो...,https://mahafood.gov.in/scheme/%e0%a4%b6%e0%a4...
3,4,https://maharashtra.gov.in/Site/1604/scheme,कृषी योजना,शेतकरी वर्गासाठी राज्य शासनातर्फे अनेक योजना उ...,https://www.manage.gov.in/fpoacademy/SGSchemes...
4,5,https://maharashtra.gov.in/Site/1604/scheme,कृषी तारण कर्ज योजना,शेतकऱ्याला असलेल्या आर्थिक गरजेपोटी तसेच स्थान...,https://www.msamb.com/Schemes/PledgeFinance


In [3]:
df.isna().sum()

scheme_id      0
site           0
scheme_name    0
description    0
scheme_link    0
dtype: int64

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('l3cube-pune/marathi-sentence-similarity-sbert')

In [5]:
df["description_vector"] = df["description"].apply(lambda x: model.encode(x))


In [6]:
df.head()

,scheme_id,site,scheme_name,description,scheme_link,description_vector
0,1,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,आनंदाचा शिधा,दि. 04.10.2022 च्या शासन निर्णयानुसार राष्ट्री...,https://mahafood.gov.in/scheme/%e0%a4%86%e0%a4...,"[-0.03430499, -0.0051293606, 0.022085818, 0.00..."
1,2,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,एपीएल शेतकरी,"राज्यातील छत्रपती संभाजीनगर, जालना, नांदेड, बी...",https://mahafood.gov.in/scheme/%e0%a4%8f%e0%a4...,"[-0.03178832, -0.01153959, 0.0040703802, 0.006..."
2,3,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,शिवभोजन,राज्यातील गरीब व गरजू जनतेला सवलतीच्या दरात भो...,https://mahafood.gov.in/scheme/%e0%a4%b6%e0%a4...,"[-0.017393228, -0.0021912777, 0.0157103, 0.018..."
3,4,https://maharashtra.gov.in/Site/1604/scheme,कृषी योजना,शेतकरी वर्गासाठी राज्य शासनातर्फे अनेक योजना उ...,https://www.manage.gov.in/fpoacademy/SGSchemes...,"[-0.019458463, -0.012561898, -0.0048517943, 0...."
4,5,https://maharashtra.gov.in/Site/1604/scheme,कृषी तारण कर्ज योजना,शेतकऱ्याला असलेल्या आर्थिक गरजेपोटी तसेच स्थान...,https://www.msamb.com/Schemes/PledgeFinance,"[-0.017723113, -0.0029795915, 0.0073545375, 0...."


In [7]:
es.ping()

True

In [35]:
from indexMappings import indexMappings
es.indices.create(index = "schemes_mapping" , mappings = indexMappings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': False, 'index': 'schemes_mapping'})

generate index mappings

insert data into index

In [8]:
record_list = df.to_dict("records")

In [9]:
record_list[0]

{'scheme_id': 1,
 'site': 'https://mahafood.gov.in/provider/%E0%A4%B0%E0%A4%BE%E0%A4%9C%E0%A5%8D%E0%A4%AF-%E0%A4%B8%E0%A4%B0%E0%A4%95%E0%A4%BE%E0%A4%B0-mr/',
 'scheme_name': 'आनंदाचा शिधा',
 'description': 'दि. 04.10.2022 च्या शासन निर्णयानुसार राष्ट्रीय अन्नसुरक्षा अधिनियम, 2013 अंतर्गत अंत्योदय अन्न योजना, प्राधान्य कुटुंब तसेच औरंगाबाद व अमरावती विभागातील सर्व जिल्हे व नागपूर विभागातील वर्धा अशा 14 शेतकरी आत्महत्याग्रस्त जिल्ह्यांतील एपीएल केशरी शेतकरी शिधापत्रिकाधारकांना ई-पॅास प्रणालीद्वारे 1 किलो रवा, 1 किलो चणाडाळ, 1 किलो साखर व 1 लिटर पामतेल या 4 शिधाजिन्नसांचा समावेश असलेल्या विशेष शिधाजिन्नस संचांचे वितरण  100-प्रति संच या दराने करण्यात आले आहे. त्यानुसार मंजूर करण्यात आलेल्या 1.61 कोटी शिधाजिन्नस संचांपैकी 1.61 कोटी शिधाजिन्नस संचांचे सुमारे 100 पात्र शिधापत्रिकाधारकांना वितरण करण्यात आले आहे.',
 'scheme_link': 'https://mahafood.gov.in/scheme/%e0%a4%86%e0%a4%a8%e0%a4%82%e0%a4%a6%e0%a4%be%e0%a4%9a%e0%a4%be-%e0%a4%b6%e0%a4%bf%e0%a4%a7%e0%a4%be/',
 'description_vector': array([

In [21]:
es.ping()

True

In [11]:
for record in record_list:
    try:
        es.index(index="schemes_mapping", document=record, id=record['scheme_id'])
    except Exception as e:
        print("error", e)

In [26]:
print(es.cluster.health())


{'cluster_name': 'elasticsearch', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 7, 'active_shards': 7, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 3, 'unassigned_primary_shards': 0, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 70.0}


In [12]:
es.count(index="schemes_mapping")

ObjectApiResponse({'count': 64, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

In [13]:
input_text = "आरोग्य"

## lexical search query

In [14]:
lexical_query = {
    "query": {
        "multi_match": {
            "query": input_text,
            "fields": ["scheme_name", "description"]
        }
    },
    "size": 5
}
result = es.search(index="marathi_schemes", body=lexical_query, source=["scheme_name", "description"])
result['hits']['hits']

C:\Users\lenovo\AppData\Local\Temp\ipykernel_27936\1820745983.py:10: DeprecationWarning: Received 'source' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  result = es.search(index="marathi_schemes", body=lexical_query, source=["scheme_name", "description"])


[{'_index': 'marathi_schemes',
  '_id': '5',
  '_score': 2.8766742,
  '_source': {'scheme_name': 'मुफ्त आरोग्य तपासणी शिबिर',
   'description': 'ग्रामीण भागात रहिवाशांसाठी विनामूल्य आरोग्य तपासणी शिबिर'}}]

## semantic search query

In [18]:

vector = model.encode(input_text)

query = {
    "size": 2,
    "knn": {
        "field": "description_vector",
        "query_vector": vector.tolist(),
        "k": 2,
        "num_candidates": 10
    }
}

result = es.search(index="schemes_mapping", body=query, source=["scheme_name", "description"])
result['hits']['hits']

C:\Users\lenovo\AppData\Local\Temp\ipykernel_27936\982302055.py:13: DeprecationWarning: Received 'source' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  result = es.search(index="schemes_mapping", body=query, source=["scheme_name", "description"])


[{'_index': 'schemes_mapping',
  '_id': '33',
  '_score': 0.8570445,
  '_source': {'scheme_name': 'राज्य शासन योजना',
   'description': 'आयुष चारित्र्य प्रमाणपत्र देणे आयुष अभ्यास प्रमाणपत्र देणे इस्सू ऑफ नो ओबजेशन सिर्तीफिकॅते आयुष ना देय प्रमाणपत्र देणे आयुष कीरकोळ जखम यांचे प्रमाणपत्र देणे आयुष कार्यमुक्त प्रमाणपत्र देणे आयुष वैद्यकीय प्रमाणपत्र देणे'}},
 {'_index': 'schemes_mapping',
  '_id': '57',
  '_score': 0.82223785,
  '_source': {'scheme_name': 'राज्य शासन योजना',
   'description': 'Changing the name of the business HealthFree Cancellation of license HealthFree Cancellation of license ProjectionFree Cancellation of license AdvertisementFree Change in Name of Business Glow SignFree Cancellation of license Glow SignFree'}}]

## hybrid search query

In [20]:
hybrid_query = {
    "size": 3,
    "query": {
        "bool": {
            "should": [
                {
                    "multi_match": {
                        "query": input_text,
                        "fields": ["scheme_name", "description"],
                        "boost": 0.4
                    }
                }
            ]
        }
    },
    "knn": {
        "field": "description_vector",
        "query_vector": vector.tolist(),
        "k": 5,
        "num_candidates": 20,
        "boost": 0.6
    }
}

result = es.search(index="marathi_schemes", body=hybrid_query, source=["scheme_name", "description"])
result['hits']['hits']

C:\Users\lenovo\AppData\Local\Temp\ipykernel_27936\1345303772.py:25: DeprecationWarning: Received 'source' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  result = es.search(index="marathi_schemes", body=hybrid_query, source=["scheme_name", "description"])


[{'_index': 'marathi_schemes',
  '_id': '5',
  '_score': 1.6685821,
  '_source': {'scheme_name': 'मुफ्त आरोग्य तपासणी शिबिर',
   'description': 'ग्रामीण भागात रहिवाशांसाठी विनामूल्य आरोग्य तपासणी शिबिर'}},
 {'_index': 'marathi_schemes',
  '_id': '22',
  '_score': 0.5126465,
  '_source': {'scheme_name': 'वृक्षारोपण अभियान',
   'description': 'जागतिक आरोग्यासाठी मोठ्या प्रमाणावर वृक्षारोपण करणे'}},
 {'_index': 'marathi_schemes',
  '_id': '29',
  '_score': 0.50772965,
  '_source': {'scheme_name': 'मोफत दवाखाना',
   'description': 'ग्रामीण भागातील रुग्णांसाठी मोफत दवाखाना आणि औषधे'}}]